<div style="background:linear-gradient(135deg,#0d0d0d,#1a252f);padding:44px 38px 36px;border-radius:12px;margin-bottom:10px">
  <h1 style="color:#fff;font-size:2.5em;margin:0 0 8px 0;font-family:Georgia,serif;letter-spacing:-0.5px">🚢 Titanic Survival Analysis</h1>
  <h3 style="color:#aaa;margin:0 0 24px 0;font-weight:normal;font-style:italic;font-family:Georgia,serif">Advanced Research-Grade EDA · Interactive Plotly · Animated Charts</h3>
  <hr style="border:none;border-top:1px solid #2c3e50;margin:0 0 22px 0">
  <table style="color:#ccc;font-size:0.88em;font-family:monospace;border-collapse:collapse">
    <tr><td style="padding:3px 8px 3px 0"><b>Dataset</b></td><td style="color:#555">│</td><td style="padding-left:10px">Titanic Passenger Manifest — 891 records · 12 features</td></tr>
    <tr><td style="padding:3px 8px 3px 0"><b>Charts</b></td><td style="color:#555">│</td><td style="padding-left:10px">Interactive Zoom · Hover · Click · Animate · Download</td></tr>
    <tr><td style="padding:3px 8px 3px 0"><b>Stats</b></td><td style="color:#555">│</td><td style="padding-left:10px">Chi-Square · Mann-Whitney U · Point-Biserial · Shapiro-Wilk</td></tr>
    <tr><td style="padding:3px 8px 3px 0"><b>Tools</b></td><td style="color:#555">│</td><td style="padding-left:10px">Python · Pandas · NumPy · SciPy · Plotly · Seaborn</td></tr>
  </table>
</div>

## Abstract
This study performs a systematic, research-grade EDA on the RMS Titanic passenger dataset. Through univariate, bivariate, multivariate, and animated temporal analysis — supported by formal hypothesis testing — we identify **gender**, **passenger class**, and **fare** as the strongest survival predictors. All findings directly inform ML preprocessing and feature engineering strategy.

---
## Table of Contents
```
01  Environment Setup
02  Data Load & Inspection
03  Data Quality & Missing Values
04  Feature Engineering
05  Descriptive Statistics (Extended)
06  Survival Overview (Interactive)
07  Categorical Feature Analysis
08  Numerical Distributions + Q-Q Plots
09  Bivariate Analysis
10  Multivariate — Scatter, Sunburst, Treemap
11  Animated Analysis
12  Statistical Hypothesis Testing
13  Outlier Detection (IQR + Visual)
14  Correlation & Pre-Modeling Importance
15  Survival Probability Curves
16  Research Findings
17  ML Preprocessing Roadmap
```
---

## 01 — Environment Setup

In [ ]:
import numpy  as np
import pandas as pd
import os, warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', 25)

from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, pointbiserialr, shapiro

import plotly.express        as px
import plotly.graph_objects  as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

import matplotlib.pyplot as plt
import seaborn           as sns

os.makedirs('plots', exist_ok=True)

# ── Global palette ───────────────────────────────────────────────
C_SURV  = '#27ae60'
C_NSURV = '#c0392b'
C_BLUE  = '#2980b9'
C_PURP  = '#8e44ad'
C_ORG   = '#e67e22'
PALETTE = {'Survived': C_SURV, 'Not Survived': C_NSURV}

TPL = 'plotly_white'

def sig_label(p):
    if   p < 0.001: return '★★★ p < 0.001'
    elif p < 0.01:  return '★★  p < 0.01'
    elif p < 0.05:  return '★   p < 0.05'
    return 'ns  p ≥ 0.05'

print('✅  Environment ready.')
print(f'    NumPy {np.__version__} | Pandas {pd.__version__}')

---
## 02 — Data Load & Inspection

In [ ]:
df_raw = pd.read_csv('data/titanic.csv')
df     = df_raw.copy()

print('━'*55)
print(f'  Rows     : {df.shape[0]:,}')
print(f'  Columns  : {df.shape[1]}')
print(f'  Memory   : {df.memory_usage(deep=True).sum()/1024:.1f} KB')
print('━'*55)
df.sample(8, random_state=42)

In [ ]:
print('FEATURE OVERVIEW')
print('─'*65)
for col in df.columns:
    dtype   = str(df[col].dtype)
    nuniq   = df[col].nunique()
    missing = df[col].isna().sum()
    sample  = str(df[col].dropna().iloc[0])[:22]
    kind    = 'Categorical' if dtype == 'object' or nuniq < 10 else 'Numerical'
    print(f'  {col:15s}  {dtype:8s}  unique={nuniq:4d}  missing={missing:3d}  [{kind}]  e.g. {sample}')

---
## 03 — Data Quality & Missing Values

In [ ]:
miss = pd.DataFrame({
    'Missing Count' : df.isnull().sum(),
    'Missing %'     : (df.isnull().mean()*100).round(2),
    'Dtype'         : df.dtypes,
    'Unique Values' : df.nunique(),
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print('MISSING VALUE AUDIT')
print('─'*60)
print(miss.to_string())
print('─'*60)
print(f'  Duplicate rows : {df.duplicated().sum()}')
print(f'  Total nulls    : {df.isnull().sum().sum()}')

In [ ]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Missing Value % per Feature',
                    'Null Pattern Heatmap (Yellow=Missing)'],
    column_widths=[0.4, 0.6])

miss_r = miss.reset_index()
miss_r.columns = ['Feature','Missing Count','Missing %','Dtype','Unique']

fig.add_trace(go.Bar(
    x=miss_r['Missing %'], y=miss_r['Feature'],
    orientation='h', text=miss_r['Missing %'].astype(str)+'%',
    textposition='outside',
    marker=dict(
        color=miss_r['Missing %'],
        colorscale='Reds', showscale=False
    ),
    hovertemplate='<b>%{y}</b><br>Missing: %{x}%<extra></extra>'
), row=1, col=1)

# Heatmap of nulls
null_matrix = df.isnull().astype(int).T
fig.add_trace(go.Heatmap(
    z=null_matrix.values,
    x=list(range(len(df))),
    y=null_matrix.index.tolist(),
    colorscale=[[0,'#ecf0f1'],[1,'#e74c3c']],
    showscale=False,
    hovertemplate='Row %{x}<br>%{y}: %{z}<extra></extra>'
), row=1, col=2)

fig.add_vline(x=50, line_dash='dash', line_color='red',
              annotation_text='50%', row=1, col=1)
fig.update_layout(title='Data Quality — Missing Value Assessment',
                  template=TPL, height=320)
fig.show()

---
## 04 — Feature Engineering

In [ ]:
df.drop_duplicates(inplace=True)

# Labels
df['Survived_Label'] = df['Survived'].map({1:'Survived', 0:'Not Survived'})
df['Class_Label']    = df['Pclass'].map({1:'1st Class',2:'2nd Class',3:'3rd Class'})
df['Port_Label']     = df['Embarked'].map({'S':'Southampton','C':'Cherbourg','Q':'Queenstown'})

# Title extraction
df['Title'] = df['Name'].str.extract(r',\s*([^.]+)\.')
rare = df['Title'].value_counts()
df['Title'] = df['Title'].replace(rare[rare < 10].index, 'Rare')

# Family
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Is_Alone']    = (df['Family_Size'] == 1).astype(int)
df['Family_Cat']  = pd.cut(df['Family_Size'], bins=[0,1,4,20],
                            labels=['Solo','Small (2-4)','Large (5+)'])

# Age
df['Age_Group'] = pd.cut(df['Age'], bins=[0,12,18,35,60,100],
                          labels=['Child','Teen','Adult','Middle-Aged','Senior'])

# Fare
df['Fare_Band'] = pd.qcut(df['Fare'], q=4,
                           labels=['Q1 Low','Q2','Q3','Q4 High'])
df['Log_Fare']  = np.log1p(df['Fare'])

# Cabin
df['Has_Cabin']   = df['Cabin'].notna().astype(int)
df['Deck']        = df['Cabin'].str[0].fillna('Unknown')

# Ticket prefix
df['Ticket_Prefix'] = df['Ticket'].str.extract(r'^([A-Za-z./]+)').fillna('Numeric')

print(f'✅  Feature Engineering — {df.shape[1]} total columns now')
print(f'    New features: Survived_Label, Class_Label, Port_Label, Title,')
print(f'                  Family_Size, Is_Alone, Family_Cat, Age_Group,')
print(f'                  Fare_Band, Log_Fare, Has_Cabin, Deck')
df[['Name','Title','Age','Age_Group','Fare','Log_Fare',
    'Family_Size','Family_Cat','Deck','Survived_Label']].sample(6, random_state=1)

---
## 05 — Descriptive Statistics (Extended)

In [ ]:
def ext_stats(series, name):
    s = series.dropna()
    Q1,Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR   = Q3-Q1
    out   = ((s < Q1-1.5*IQR)|(s > Q3+1.5*IQR)).sum()
    _, sw_p = shapiro(s.sample(min(len(s),500), random_state=42))
    return {'Feature':name, 'N':len(s), 'Missing':series.isna().sum(),
            'Mean':round(s.mean(),3), 'Median':round(s.median(),3),
            'Std':round(s.std(),3), 'Min':round(s.min(),3), 'Max':round(s.max(),3),
            'Skew':round(s.skew(),4), 'Kurt':round(s.kurtosis(),4),
            'IQR':round(IQR,3), 'Outliers':out, 'Shapiro-p':round(sw_p,4)}

st_df = pd.DataFrame([
    ext_stats(df['Age'],'Age'),
    ext_stats(df['Fare'],'Fare'),
    ext_stats(df['Log_Fare'],'Log_Fare'),
    ext_stats(df['SibSp'],'SibSp'),
    ext_stats(df['Parch'],'Parch'),
    ext_stats(df['Family_Size'],'Family_Size'),
]).set_index('Feature')

print('EXTENDED DESCRIPTIVE STATISTICS')
print('─'*100)
print(st_df.to_string())
print('─'*100)
print('Notes: |Skew|>1=highly skewed | Kurt>3=heavy tails | Shapiro-p<0.05=non-normal')

In [ ]:
# Interactive stats table
st_r = st_df.reset_index()
fig = go.Figure(data=[go.Table(
    columnwidth=[120]+[80]*len(st_df.columns),
    header=dict(
        values=['<b>'+c+'</b>' for c in st_r.columns],
        fill_color='#2c3e50', font=dict(color='white',size=11),
        align='center', height=34
    ),
    cells=dict(
        values=[st_r[c].tolist() for c in st_r.columns],
        fill_color=[['#f0f3f4' if i%2==0 else 'white'
                     for i in range(len(st_r))]]*len(st_r.columns),
        align='center', height=28, font=dict(size=10)
    )
)])
fig.update_layout(title='Descriptive Statistics Table',
                  template=TPL, height=280)
fig.show()

In [ ]:
# Categorical frequency
print('CATEGORICAL FREQUENCY TABLE')
print('─'*60)
for col in ['Survived_Label','Sex','Class_Label','Port_Label','Title','Family_Cat','Age_Group']:
    vc  = df[col].value_counts(dropna=False)
    pct = (vc / len(df) * 100).round(1)
    print(f'\n  {col}')
    for val, cnt in vc.items():
        bar = '█' * max(1, int(pct[val]/3))
        print(f'    {str(val):22s}  {cnt:4d}  ({pct[val]:5.1f}%)  {bar}')

---
## 06 — Survival Overview (Interactive)

In [ ]:
vc = df['Survived_Label'].value_counts().reindex(['Survived','Not Survived'])

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Count','Percentage','Donut Chart'],
    specs=[[{'type':'bar'},{'type':'bar'},{'type':'pie'}]]
)

fig.add_trace(go.Bar(
    x=vc.index, y=vc.values,
    marker_color=[C_SURV, C_NSURV],
    text=vc.values, textposition='outside',
    hovertemplate='<b>%{x}</b>: %{y} passengers<extra></extra>'
), row=1, col=1)

pct_vals = (vc / len(df) * 100).round(1)
fig.add_trace(go.Bar(
    x=pct_vals.index, y=pct_vals.values,
    marker_color=[C_SURV, C_NSURV],
    text=pct_vals.values.astype(str)+'%',
    textposition='outside',
    hovertemplate='<b>%{x}</b>: %{y}%<extra></extra>'
), row=1, col=2)

fig.add_trace(go.Pie(
    labels=vc.index, values=vc.values,
    marker_colors=[C_SURV, C_NSURV],
    hole=0.5, textinfo='label+percent',
    hovertemplate='%{label}: %{value} (%{percent})<extra></extra>'
), row=1, col=3)

fig.update_layout(
    title='🚢 Overall Survival Distribution — Titanic',
    template=TPL, height=420, showlegend=False
)
fig.show()

print(f'\n  Survived     : {vc["Survived"]:3d}  ({pct_vals["Survived"]:.1f}%)')
print(f'  Not Survived : {vc["Not Survived"]:3d}  ({pct_vals["Not Survived"]:.1f}%)')
print(f'  → Class imbalance ratio 1:{vc["Not Survived"]/vc["Survived"]:.2f}')

---
## 07 — Categorical Feature Analysis

In [ ]:
cat_configs = [
    ('Sex',        None,                                     'Gender'),
    ('Class_Label',['1st Class','2nd Class','3rd Class'],    'Passenger Class'),
    ('Port_Label', ['Southampton','Cherbourg','Queenstown'], 'Port of Embarkation'),
    ('Title',      None,                                     'Passenger Title'),
    ('Family_Cat', ['Solo','Small (2-4)','Large (5+)'],      'Family Category'),
    ('Age_Group',  ['Child','Teen','Adult','Middle-Aged','Senior'], 'Age Group'),
    ('Fare_Band',  ['Q1 Low','Q2','Q3','Q4 High'],           'Fare Band'),
    ('Has_Cabin',  [0, 1],                                   'Has Cabin'),
]

for col, order, label in cat_configs:
    surv = df.groupby(col, observed=True).agg(
        Rate=('Survived','mean'),
        Count=('Survived','count'),
        Survived_n=('Survived','sum')
    ).reset_index()
    surv['Rate'] = (surv['Rate']*100).round(1)
    surv['Not_Survived_n'] = surv['Count'] - surv['Survived_n']

    if order:
        valid_order = [o for o in order if o in surv[col].values]
        surv[col] = pd.Categorical(surv[col].astype(str), categories=[str(o) for o in valid_order], ordered=True)
        surv = surv.sort_values(col)
    else:
        surv = surv.sort_values('Rate', ascending=False)
    surv[col] = surv[col].astype(str)

    fig = make_subplots(rows=1, cols=2,
        subplot_titles=[f'Survival Rate (%) by {label}',
                        f'Survived vs Not Survived by {label}'])

    fig.add_trace(go.Bar(
        x=surv[col], y=surv['Rate'],
        marker=dict(color=surv['Rate'],
                    colorscale=[[0,C_NSURV],[0.5,'#f39c12'],[1,C_SURV]],
                    showscale=False),
        text=surv['Rate'].astype(str)+'%', textposition='outside',
        customdata=surv['Count'],
        hovertemplate='<b>%{x}</b><br>Rate: %{y}%<br>Total: %{customdata}<extra></extra>',
        name='Survival %'
    ), row=1, col=1)
    fig.add_hline(y=38.4, line_dash='dash', line_color='gray',
                  annotation_text='Overall 38.4%', row=1, col=1)

    fig.add_trace(go.Bar(
        x=surv[col], y=surv['Survived_n'],
        name='Survived', marker_color=C_SURV,
        hovertemplate='<b>%{x}</b><br>Survived: %{y}<extra></extra>'
    ), row=1, col=2)
    fig.add_trace(go.Bar(
        x=surv[col], y=surv['Not_Survived_n'],
        name='Not Survived', marker_color=C_NSURV,
        hovertemplate='<b>%{x}</b><br>Not Survived: %{y}<extra></extra>'
    ), row=1, col=2)

    fig.update_layout(
        title=f'Analysis: {label}',
        barmode='group', template=TPL,
        height=400, yaxis_range=[0,110]
    )
    fig.show()

---
## 08 — Numerical Distributions + Q-Q Plots

In [ ]:
for feat, color, unit in [
    ('Age',      C_BLUE, 'years'),
    ('Fare',     C_PURP, '£ GBP'),
    ('Log_Fare', C_ORG,  'log(£+1)'),
]:
    data = df[feat].dropna()

    fig = make_subplots(rows=1, cols=4,
        subplot_titles=['Histogram','KDE Overlay','Box + Strip','Q-Q Plot'])

    # Histogram
    fig.add_trace(go.Histogram(
        x=data, nbinsx=40, marker_color=color,
        opacity=0.8, name=feat,
        hovertemplate=f'{feat}: %{{x:.1f}}<br>Count: %{{y}}<extra></extra>'
    ), row=1, col=1)
    fig.add_vline(x=data.mean(),   line_dash='dash', line_color='red',
                  annotation_text=f'Mean={data.mean():.1f}', row=1, col=1)
    fig.add_vline(x=data.median(), line_dash='dot', line_color='black',
                  annotation_text=f'Med={data.median():.1f}', row=1, col=1)

    # KDE by survival
    for label, lcolor in [('Survived',C_SURV),('Not Survived',C_NSURV)]:
        sub = df[df['Survived_Label']==label][feat].dropna()
        kde_x = np.linspace(data.min(), data.max(), 300)
        kde   = stats.gaussian_kde(sub)(kde_x)
        fig.add_trace(go.Scatter(
            x=kde_x, y=kde, mode='lines',
            name=label, line=dict(color=lcolor, width=2.5),
            fill='tozeroy', fillcolor=lcolor.replace(')',',0.15)').replace('rgb','rgba') if lcolor.startswith('rgb') else lcolor+'26',
            hovertemplate=f'<b>{label}</b><br>{feat}: %{{x:.1f}}<extra></extra>'
        ), row=1, col=2)

    # Box
    fig.add_trace(go.Box(
        y=data, name=feat, marker_color=color,
        boxmean='sd', boxpoints='outliers',
        hovertemplate=f'{feat}: %{{y:.2f}}<extra></extra>'
    ), row=1, col=3)

    # Q-Q Plot
    (osm, osr), (slope, intercept, r) = stats.probplot(data, dist='norm')
    fig.add_trace(go.Scatter(
        x=list(osm), y=list(osr), mode='markers',
        marker=dict(color=color, size=4, opacity=0.5),
        name='Q-Q Points',
        hovertemplate='Theoretical: %{x:.2f}<br>Sample: %{y:.2f}<extra></extra>'
    ), row=1, col=4)
    fig.add_trace(go.Scatter(
        x=list(osm),
        y=[slope*x+intercept for x in osm],
        mode='lines', line=dict(color='red', width=2),
        name=f'R²={r**2:.3f}'
    ), row=1, col=4)

    skew = data.skew()
    kurt = data.kurtosis()
    _, sw_p = shapiro(data.sample(min(len(data),500), random_state=42))

    fig.update_layout(
        title=f'Distribution Analysis — {feat} [{unit}]  '
              f'| Skew={skew:.3f} | Kurt={kurt:.3f} | '
              f'Shapiro p={sw_p:.4f} ({"Non-Normal" if sw_p<0.05 else "Normal"})',
        template=TPL, height=380, showlegend=False
    )
    fig.show()
    print(f'  {feat}: N={len(data)} | Skew={skew:.4f} | Kurt={kurt:.4f} | '
          f'Shapiro p={sw_p:.6f} | Outliers={((data<data.quantile(0.25)-1.5*(data.quantile(0.75)-data.quantile(0.25)))|(data>data.quantile(0.75)+1.5*(data.quantile(0.75)-data.quantile(0.25)))).sum()}\n')

---
## 09 — Bivariate Analysis

In [ ]:
# All numerical vs survival
num_feats = [
    ('Age','years'), ('Fare','£'), ('Log_Fare','log £'), ('Family_Size','persons')
]

for feat, unit in num_feats:
    fig = make_subplots(rows=1, cols=3,
        subplot_titles=[f'KDE Overlay','Violin Plot','Box Plot'],
        shared_yaxes=False)

    for label, lc in [('Survived',C_SURV),('Not Survived',C_NSURV)]:
        sub = df[df['Survived_Label']==label][feat].dropna()
        kde_x = np.linspace(df[feat].dropna().min(), df[feat].dropna().max(), 300)
        kde   = stats.gaussian_kde(sub)(kde_x)

        fig.add_trace(go.Scatter(
            x=kde_x, y=kde, mode='lines',
            name=label, line=dict(color=lc,width=2.5),
            fill='tozeroy', opacity=0.6
        ), row=1, col=1)

        fig.add_trace(go.Violin(
            y=sub, name=label, side='positive' if label=='Survived' else 'negative',
            line_color=lc, fillcolor=lc, opacity=0.6,
            meanline_visible=True, box_visible=True,
            points='outliers'
        ), row=1, col=2)

        fig.add_trace(go.Box(
            y=sub, name=label, marker_color=lc,
            boxmean='sd', boxpoints='outliers'
        ), row=1, col=3)

    fig.update_layout(
        title=f'{feat} [{unit}] vs Survival Status',
        template=TPL, height=400, violinmode='overlay'
    )
    fig.show()

---
## 10 — Multivariate Analysis

In [ ]:
# Gender × Class × Survival heatmap
pivot = df.pivot_table(values='Survived', index='Sex',
                       columns='Class_Label', aggfunc='mean') * 100
pivot = pivot[['1st Class','2nd Class','3rd Class']].round(1)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Survival Rate (%) Heatmap','Grouped Bar Chart'],
    specs=[[{'type':'heatmap'},{'type':'bar'}]])

fig.add_trace(go.Heatmap(
    z=pivot.values, x=pivot.columns.tolist(), y=pivot.index.tolist(),
    colorscale='RdYlGn', zmin=0, zmax=100,
    text=[[f'{v:.1f}%' for v in row] for row in pivot.values],
    texttemplate='%{text}', textfont={'size':16,'color':'black'},
    hovertemplate='<b>%{y} · %{x}</b><br>Survival: %{z:.1f}%<extra></extra>',
    showscale=True
), row=1, col=1)

gx = df.groupby(['Class_Label','Sex'])['Survived'].mean().reset_index()
gx.columns = ['Class','Sex','Rate']
gx['Rate'] = (gx['Rate']*100).round(1)
gx = gx[gx['Class'].isin(['1st Class','2nd Class','3rd Class'])]
for sex, color in [('male',C_BLUE),('female','#e91e8c')]:
    sub = gx[gx['Sex']==sex]
    fig.add_trace(go.Bar(
        x=sub['Class'], y=sub['Rate'], name=sex,
        marker_color=color,
        text=sub['Rate'].astype(str)+'%', textposition='outside'
    ), row=1, col=2)

fig.update_layout(title='Gender × Class × Survival',
                  template=TPL, height=420, barmode='group')
fig.show()

In [ ]:
# Scatter: Age vs Fare by Class & Survival
df_sc = df.dropna(subset=['Age'])
fig = px.scatter(
    df_sc, x='Age', y='Fare',
    color='Survived_Label',
    symbol='Sex',
    size='Log_Fare',
    facet_col='Class_Label',
    facet_col_order=['1st Class','2nd Class','3rd Class'],
    color_discrete_map=PALETTE,
    hover_data=['Age','Fare','Sex','Class_Label','Survived_Label','Title'],
    title='Age vs Fare — Split by Class · Colored by Survival',
    template=TPL, height=480
)
fig.update_traces(opacity=0.65)
fig.show()

In [ ]:
# Sunburst + Treemap
df_sun = df.groupby(['Class_Label','Sex','Survived_Label']).size().reset_index(name='Count')

fig1 = px.sunburst(df_sun,
    path=['Class_Label','Sex','Survived_Label'],
    values='Count',
    color='Survived_Label',
    color_discrete_map={**PALETTE,'(?)':'#bdc3c7'},
    title='Sunburst — Class → Gender → Survival',
    template=TPL, height=500)
fig1.show()

fig2 = px.treemap(df_sun,
    path=['Class_Label','Sex','Survived_Label'],
    values='Count',
    color='Survived_Label',
    color_discrete_map={**PALETTE,'(?)':'#bdc3c7'},
    title='Treemap — Class → Gender → Survival',
    template=TPL, height=480)
fig2.show()

---
## 11 — Animated Analysis

In [ ]:
# Animated bar: survival rate by age group across classes
df_anim = df.dropna(subset=['Age_Group'])
anim_data = df_anim.groupby(['Class_Label','Age_Group'], observed=True).agg(
    Rate=('Survived','mean'), Count=('Survived','count')
).reset_index()
anim_data['Rate']      = (anim_data['Rate']*100).round(1)
anim_data['Age_Group'] = anim_data['Age_Group'].astype(str)

fig = px.bar(
    anim_data,
    x='Age_Group', y='Rate',
    color='Rate',
    animation_frame='Class_Label',
    animation_group='Age_Group',
    color_continuous_scale=[[0,C_NSURV],[0.5,'#f39c12'],[1,C_SURV]],
    range_y=[0,100],
    text='Rate',
    custom_data=['Count'],
    title='🎬 Animated: Survival Rate by Age Group (Animate by Class)',
    labels={'Rate':'Survival Rate %','Age_Group':'Age Group'},
    category_orders={'Age_Group':['Child','Teen','Adult','Middle-Aged','Senior']},
    template=TPL, height=460
)
fig.update_traces(
    texttemplate='%{text}%', textposition='outside',
    hovertemplate='<b>%{x}</b><br>Rate: %{y}%<br>N: %{customdata[0]}<extra></extra>'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Animated scatter: Age vs LogFare across classes
df_as = df.dropna(subset=['Age']).copy()
df_as['Class_Label'] = pd.Categorical(
    df_as['Class_Label'],
    categories=['1st Class','2nd Class','3rd Class'], ordered=True
)
df_as = df_as.sort_values('Class_Label')

fig = px.scatter(
    df_as, x='Age', y='Log_Fare',
    color='Survived_Label',
    animation_frame='Class_Label',
    animation_group='PassengerId',
    size='Family_Size',
    color_discrete_map=PALETTE,
    hover_data=['Sex','Title','Age','Fare','Survived_Label'],
    range_x=[0,85], range_y=[-0.5,7],
    title='🎬 Animated Scatter: Age vs Log(Fare) by Passenger Class',
    labels={'Log_Fare':'Log(Fare)','Survived_Label':'Status'},
    template=TPL, height=480
)
fig.update_traces(opacity=0.7)
fig.show()

---
## 12 — Statistical Hypothesis Testing

In [ ]:
results = []

tests = [
    ('Chi-Square', 'Sex vs Survived',
     lambda: chi2_contingency(pd.crosstab(df['Sex'],df['Survived']))[:2]),
    ('Chi-Square', 'Pclass vs Survived',
     lambda: chi2_contingency(pd.crosstab(df['Pclass'],df['Survived']))[:2]),
    ('Chi-Square', 'Embarked vs Survived',
     lambda: chi2_contingency(pd.crosstab(df.dropna(subset=['Embarked'])['Embarked'],
                              df.dropna(subset=['Embarked'])['Survived']))[:2]),
    ('Mann-Whitney U', 'Age: Survived vs Not',
     lambda: mannwhitneyu(df[df['Survived']==1]['Age'].dropna(),
                          df[df['Survived']==0]['Age'].dropna(),
                          alternative='two-sided')),
    ('Mann-Whitney U', 'Fare: Survived vs Not',
     lambda: mannwhitneyu(df[df['Survived']==1]['Fare'],
                          df[df['Survived']==0]['Fare'],
                          alternative='two-sided')),
    ('Mann-Whitney U', 'Family_Size: Survived vs Not',
     lambda: mannwhitneyu(df[df['Survived']==1]['Family_Size'],
                          df[df['Survived']==0]['Family_Size'],
                          alternative='two-sided')),
    ('Point-Biserial', 'Fare correlation w/ Survived',
     lambda: pointbiserialr(df['Survived'],df['Fare'])),
]

for test, variable, fn in tests:
    stat, p = fn()
    results.append({'Test':test,'Variable':variable,
                    'Statistic':round(stat,4),'p-value':round(p,8),
                    'Significance':sig_label(p)})

res_df = pd.DataFrame(results)
print('HYPOTHESIS TEST RESULTS  (α = 0.05)')
print('─'*85)
print(res_df.to_string(index=False))
print('─'*85)

In [ ]:
res_df['neg_log_p'] = -np.log10(res_df['p-value'].astype(float).clip(1e-300))

fig = px.bar(
    res_df.sort_values('neg_log_p'),
    x='neg_log_p', y='Variable',
    orientation='h',
    color='neg_log_p',
    color_continuous_scale='RdYlGn',
    text='Significance',
    hover_data={'Test':True,'Statistic':True,'p-value':True,'neg_log_p':False},
    title='Statistical Significance — -log₁₀(p-value)',
    labels={'neg_log_p':'-log₁₀(p)','Variable':''},
    template=TPL, height=420
)
fig.add_vline(x=-np.log10(0.05),  line_dash='dash', line_color='navy',
              annotation_text='α=0.05')
fig.add_vline(x=-np.log10(0.001), line_dash='dot',  line_color='red',
              annotation_text='α=0.001')
fig.update_traces(textposition='inside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

---
## 13 — Outlier Detection

In [ ]:
def iqr_report(series, name):
    s = series.dropna()
    Q1,Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR   = Q3-Q1
    lo,hi = Q1-1.5*IQR, Q3+1.5*IQR
    mask  = (s<lo)|(s>hi)
    return {'Feature':name,'Q1':round(Q1,2),'Q3':round(Q3,2),
            'IQR':round(IQR,2),'Lower Fence':round(lo,2),'Upper Fence':round(hi,2),
            'Outliers':mask.sum(),'Outlier %':round(mask.mean()*100,2)}

out_df = pd.DataFrame([
    iqr_report(df['Age'],'Age'),
    iqr_report(df['Fare'],'Fare'),
    iqr_report(df['Log_Fare'],'Log_Fare'),
    iqr_report(df['Family_Size'],'Family_Size'),
    iqr_report(df['SibSp'],'SibSp'),
    iqr_report(df['Parch'],'Parch'),
]).set_index('Feature')

print('IQR OUTLIER REPORT')
print('─'*80)
print(out_df.to_string())
print('─'*80)

In [ ]:
# Violin + box for all numerical features
num_cols   = ['Age','Fare','Log_Fare','Family_Size','SibSp','Parch']
col_colors = [C_BLUE, C_PURP, C_ORG, '#1abc9c', C_NSURV, '#f39c12']

fig = make_subplots(rows=2, cols=3, subplot_titles=num_cols)

for i, (col, color) in enumerate(zip(num_cols, col_colors)):
    row, c = divmod(i, 3)
    data = df[col].dropna()
    Q1,Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR   = Q3-Q1
    out_mask = (data < Q1-1.5*IQR)|(data > Q3+1.5*IQR)

    fig.add_trace(go.Violin(
        y=data, name=col,
        box_visible=True, meanline_visible=True,
        fillcolor=color, opacity=0.55,
        line_color='#2c3e50', points='outliers',
        marker=dict(color=C_NSURV, size=4),
        hovertemplate=f'<b>{col}</b>: %{{y:.2f}}<extra></extra>'
    ), row=row+1, col=c+1)

fig.update_layout(
    title='Outlier Detection — Violin + Box (All Numerical Features)',
    template=TPL, height=580, showlegend=False
)
fig.show()

---
## 14 — Correlation & Pre-Modeling Importance

In [ ]:
df_enc = df.copy()
df_enc['Sex_Enc']      = (df_enc['Sex']=='female').astype(int)
df_enc['Embarked_Enc'] = df_enc['Embarked'].map({'S':0,'C':1,'Q':2})

corr_cols = ['Survived','Pclass','Sex_Enc','Age','SibSp','Parch',
             'Fare','Log_Fare','Family_Size','Is_Alone','Has_Cabin','Embarked_Enc']

corr = df_enc[corr_cols].corr().round(3)

# Full heatmap
fig = px.imshow(corr, text_auto=True,
                color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1,
                title='Full Feature Correlation Heatmap',
                aspect='auto', template=TPL, height=560)
fig.update_traces(texttemplate='%{z:.2f}', textfont_size=9)
fig.show()

# Bar: correlation with Survived
surv_corr = corr['Survived'].drop('Survived').sort_values()
fig2 = px.bar(
    x=surv_corr.values, y=surv_corr.index,
    orientation='h',
    color=surv_corr.values,
    color_continuous_scale='RdYlGn',
    text=[f'{v:.3f}' for v in surv_corr.values],
    title='Feature Correlation with Survived (Ranked)',
    labels={'x':'Pearson r','y':''},
    template=TPL, height=420
)
fig2.add_vline(x=0, line_color='black', line_width=1.5)
fig2.update_layout(coloraxis_showscale=False)
fig2.show()

---
## 15 — Survival Probability Curves

In [ ]:
# Survival probability by Age (rolling window)
df_age = df.dropna(subset=['Age']).sort_values('Age')

fig = go.Figure()

for cls, color in [('1st Class','#f1c40f'),('2nd Class','#95a5a6'),('3rd Class','#cd7f32')]:
    sub = df_age[df_age['Class_Label']==cls]
    if len(sub) < 10: continue
    # Rolling 20-passenger window
    sub = sub.sort_values('Age')
    roll_surv = sub['Survived'].rolling(window=20, min_periods=5).mean() * 100
    fig.add_trace(go.Scatter(
        x=sub['Age'], y=roll_surv,
        mode='lines', name=cls,
        line=dict(color=color, width=3),
        fill='tozeroy', opacity=0.15,
        hovertemplate=f'<b>{cls}</b><br>Age: %{{x:.1f}}<br>Survival prob: %{{y:.1f}}%<extra></extra>'
    ))

fig.add_hline(y=38.4, line_dash='dash', line_color='gray',
              annotation_text='Overall 38.4%')
fig.update_layout(
    title='Survival Probability Curve by Age — Split by Passenger Class',
    xaxis_title='Age (years)',
    yaxis_title='Survival Probability (%)',
    template=TPL, height=440
)
fig.show()

In [ ]:
# Survival probability by Fare (rolling)
df_fare = df.sort_values('Fare')

fig2 = go.Figure()
for sex, color in [('male', C_BLUE),('female','#e91e8c')]:
    sub = df_fare[df_fare['Sex']==sex].sort_values('Fare')
    roll = sub['Survived'].rolling(window=25, min_periods=5).mean() * 100
    fig2.add_trace(go.Scatter(
        x=sub['Fare'], y=roll,
        mode='lines', name=sex.capitalize(),
        line=dict(color=color, width=3),
        hovertemplate=f'<b>{sex}</b><br>Fare: £%{{x:.1f}}<br>Survival: %{{y:.1f}}%<extra></extra>'
    ))

fig2.add_hline(y=38.4, line_dash='dash', line_color='gray')
fig2.update_layout(
    title='Survival Probability by Fare — Male vs Female',
    xaxis_title='Fare (£)', xaxis_range=[0,300],
    yaxis_title='Survival Probability (%)',
    template=TPL, height=420
)
fig2.show()

---
## 16 — Research Findings

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║            RESEARCH FINDINGS — TITANIC SURVIVAL ANALYSIS                ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  1  GENDER — Dominant Predictor  (χ²=260.7, p<0.001)                  ║
║     Female ~74% survived  vs  Male ~19%                                 ║
║     'Women and children first' statistically confirmed.                 ║
║                                                                          ║
║  2  PASSENGER CLASS  (χ²=102.9, p<0.001)                               ║
║     1st: 63%  |  2nd: 47%  |  3rd: 24%                                 ║
║     Lifeboat proximity + socioeconomic prioritization.                  ║
║                                                                          ║
║  3  FARE — Socioeconomic Proxy  (p<0.001, r=0.257)                     ║
║     Higher fare → higher survival. Skew=4.79 → log transform needed.  ║
║                                                                          ║
║  4  CLASS IMBALANCE  (38.4% survived)                                   ║
║     Raw accuracy is misleading. Use ROC-AUC + F1-score.                ║
║                                                                          ║
║  5  AGE  (p<0.05)                                                        ║
║     Child ~58%  |  Senior ~27%.  19.9% missing.                        ║
║                                                                          ║
║  6  FAMILY SIZE  (p<0.01)                                               ║
║     Small (2-4): best survival. Non-linear → bin as categorical.        ║
║                                                                          ║
║  7  CABIN — Hidden Signal                                               ║
║     Has_Cabin=1: ~67%  vs  0: ~30%. Raw=77% missing.                  ║
║                                                                          ║
║  8  EMBARKED — Indirect via Class                                       ║
║     Cherbourg 55% vs Southampton 34% — class composition effect.       ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

---
## 17 — ML Preprocessing Roadmap

In [ ]:
roadmap = {
    'Missing Values'   : ['Age → KNN or median by Pclass+Sex',
                          'Cabin → drop raw, keep Has_Cabin flag',
                          'Embarked → mode imputation (2 rows only)'],
    'Encoding'         : ['Sex → binary (female=1)',
                          'Embarked → One-Hot',
                          'Pclass → ordinal int (keep as-is)'],
    'Transformation'   : ['Fare → np.log1p(Fare) skew 4.79→0.5',
                          'Age → StandardScaler after imputation'],
    'Keep Features'    : ['Family_Size, Is_Alone, Family_Cat',
                          'Has_Cabin, Title, Log_Fare, Deck'],
    'Drop Features'    : ['PassengerId, Name, Ticket, Cabin (raw)'],
    'Class Imbalance'  : ['class_weight="balanced" in sklearn',
                          'SMOTE (training only)',
                          'Metric: ROC-AUC + F1-score'],
    'Models'           : ['Logistic Regression (baseline)',
                          'Random Forest, XGBoost, LightGBM',
                          '5-fold Stratified CV'],
}

print('ML PREPROCESSING ROADMAP')
print('═'*55)
for section, items in roadmap.items():
    print(f'\n  ▸ {section}')
    for item in items:
        print(f'      → {item}')

print('\n✅  EDA Complete — 17 sections | All charts interactive')